# Label cell types using CellTypist Models

To build our reference, we would like to start with labels that originate from published cell type references. 

One of the approaches for this cell type labeling is CellTypist, a model-based approach to cell type labeling.  

CellTypist is described [on their website](https://www.celltypist.org/), and in this publication:  

Domínguez Conde, C. et al. Cross-tissue immune cell analysis reveals tissue-specific features in humans. Science 376, eabl5197 (2022)

Here, we'll load in our cells individually, and assign labels based on our 3-level annotated PBMC reference:  

- AIFI_L1:
    - 9 types
- AIFI_L2:  
    - 29 types
- AIFI_L3:
    - 71 types

## Load Packages

`anndata`: Data structures for scRNA-seq  
`celltypist`: Model-based cell type annotation  
`concurrent.futures`: parallelization methods  
`datetime`: date and time functions  
`h5py`: HDF5 file I/O  
`hisepy`: The HISE SDK for Python  
`numpy`: Mathematical data structures and computation  
`os`: operating system calls  
`pandas`: DataFrame data structures  
`re`: Regular expressions  
`scanpy`: scRNA-seq analysis  
`scipy.sparse`: Spare matrix data structures  
`shutil`: Shell utilities

In [1]:
import anndata
import celltypist
from celltypist import models
import concurrent.futures
from datetime import date
import h5py
import hisepy
import numpy as np
import os
import pandas as pd 
import re
import scanpy as sc
import scipy.sparse as scs
import shutil

Load a model to prevent CellTypist from loading all models per core

In [2]:
models.download_models(
    force_update = True,
    model = ['Immune_All_High.pkl']
)

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 50
📂 Storing models in /root/.celltypist/data/models
💾 Total models to download: 1
💾 Downloading model [1/1]: Immune_All_High.pkl


## Helper functions

This function allows easy reading of .csv files stored in HISE

In [3]:
def read_csv_uuid(csv_uuid):
    csv_path = '/home/jupyter/cache/{u}'.format(u = csv_uuid)
    if not os.path.isdir(csv_path):
        hise_res = hisepy.reader.cache_files([csv_uuid])
    csv_filename = os.listdir(csv_path)[0]
    csv_file = '{p}/{f}'.format(p = csv_path, f = csv_filename)
    df = pd.read_csv(csv_file, index_col = 0)
    return df

This function allows easy identification of the cached file path for files retrieved from HISE

In [4]:
def read_path_uuid(file_uuid):
    file_path = '/home/jupyter/cache/{u}'.format(u = file_uuid)
    if not os.path.isdir(file_path):
        hise_res = hisepy.reader.cache_files([file_uuid])
    filename = os.listdir(file_path)[0]
    full_path = '{p}/{f}'.format(p = file_path, f = filename)
    return full_path

These functions will retrieve data for a sample, assemble an AnnData object

In [5]:
# define a function to read count data
def read_mat(h5_con):
    mat = scs.csc_matrix(
        (h5_con['matrix']['data'][:], # Count values
         h5_con['matrix']['indices'][:], # Row indices
         h5_con['matrix']['indptr'][:]), # Pointers for column positions
        shape = tuple(h5_con['matrix']['shape'][:]) # Matrix dimensions
    )
    return mat

# define a function to read obeservation metadata (i.e. cell metadata)
def read_obs(h5con):
    bc = h5con['matrix']['barcodes'][:]
    bc = [x.decode('UTF-8') for x in bc]

    # Initialized the DataFrame with cell barcodes
    obs_df = pd.DataFrame({ 'barcodes' : bc })

    # Get the list of available metadata columns
    obs_columns = h5con['matrix']['observations'].keys()

    # For each column
    for col in obs_columns:
        # Read the values
        values = h5con['matrix']['observations'][col][:]
        # Check for byte storage
        if(isinstance(values[0], (bytes, bytearray))):
            # Decode byte strings
            values = [x.decode('UTF-8') for x in values]
        # Add column to the DataFrame
        obs_df[col] = values

    obs_df = obs_df.set_index('barcodes', drop = False)
    
    return obs_df

# define a function to construct anndata object from a h5 file
def read_h5_anndata(h5_con):
    #h5_con = h5py.File(h5_file, mode = 'r')
    # extract the expression matrix
    mat = read_mat(h5_con)
    # extract gene names
    genes = h5_con['matrix']['features']['name'][:]
    genes = [x.decode('UTF-8') for x in genes]
    # extract metadata
    obs_df = read_obs(h5_con)
    # construct anndata
    adata = anndata.AnnData(mat.T,
                             obs = obs_df)
    # make sure the gene names aligned
    adata.var_names = genes

    adata.var_names_make_unique()
    return adata

This function retrieves and assembles an anndata based on the UUID for a .h5 file.

In [6]:
def get_adata(uuid):
    # Load the file using HISE
    res = hisepy.reader.read_files([uuid])

    # If there's an error, read_files returns a list instead of a dictionary.
    # We should raise and exception with the message when this happens.
    if(isinstance(res, list)):
        error_message = res[0]['message']
        raise Exception('{u}: {e}'.format(u = uuid, e = error_message))
    
    # Read the file to adata
    h5_con = res['values'][0]
    adata = read_h5_anndata(h5_con)
    
    # Close the file now that we're done with it
    h5_con.close()

    return(adata)

This function applies cell type predictions for a specific model and generates output files.

In [7]:
def run_prediction(adata, model, model_name, out_dir = "output"):
    # Make output directories
    model_dir = "{d}/{m}".format(d = out_dir, m = model_name)
    if not os.path.isdir(model_dir):
        os.makedirs(model_dir)
    
    sample_id = adata.obs['pbmc_sample_id'].unique()[0]
    label_file = "{d}/{s}_{m}_labels.csv".format(d = model_dir, s = sample_id, m = model_name)

    if os.path.exists(label_file):
        print("{s}: {m} Previously computed; Skipping.".format(s = sample_id, m = model_name))
    else:
        # Perform prediction
        predictions = celltypist.annotate(
            adata, 
            model = model, 
            majority_voting = True)
    
        # Write output
        
        prob_file = "{d}/{s}_{m}_probability_mat.parquet".format(d = model_dir, s = sample_id, m = model_name)
        prob = predictions.probability_matrix
        prob.to_parquet(prob_file)
    
        dec_file = "{d}/{s}_{m}_decision_mat.parquet".format(d = model_dir, s = sample_id, m = model_name)
        predictions.decision_matrix.to_parquet(dec_file)
        
        labels = predictions.predicted_labels
        labels = labels.rename({'predicted_labels': model_name}, axis = 1)
        
        prob_scores = []
        for i in range(labels.shape[0]):
            prob_scores.append(prob.loc[labels.index.to_list()[i],labels[model_name][i]])
        labels['{m}_score'.format(m = model_name)] = prob_scores
        labels.to_csv(label_file)

This wrapper puts the above steps together: reading data based on a UUID, normalizing the data, and labeling with all of the models in model_paths

In [8]:
def process_data(file_uuid, sample_id, model_paths):
    out_dir = "output"
    check_file = '{d}/{m}/{s}_{m}_labels.csv'.format(d = out_dir, m = 'AIFI_L3', s = sample_id)

    if os.path.exists(check_file):
        print('{s} Previously labeled; Skipping.'.format(s = sample_id))
    else:
        # Load cells from HISE .h5 files
        adata = get_adata(file_uuid)
        
        # Normalize data
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        adata.obs.index = adata.obs['barcodes']
        
        # Predict cell types
        for model_name,model_path in model_paths.items():
            run_prediction(
                adata,
                model_path,
                model_name,
                out_dir
            )
        
        del adata

This function is used to generate a unique identifier for the notebook results to help with searching for them in HISE.

In [9]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Obtain CellTypist Models

In [10]:
model_uuids = {
    'AIFI_L1': '482b9ec5-8631-48a1-a7ef-94e23fe97068',
    'AIFI_L2': 'cc78f20a-b962-4ddf-ae93-47d58741a644',
    'AIFI_L3': '671d1e43-bd32-4fea-bdda-d19a0484e664'
}

In [11]:
model_paths = {}
for name,uuid in model_uuids.items():
    model_paths[name] = read_path_uuid(uuid)

In [12]:
model_paths

{'AIFI_L1': '/home/jupyter/cache/482b9ec5-8631-48a1-a7ef-94e23fe97068/ref_pbmc_clean_celltypist_model_AIFI_L1_2024-04-18.pkl',
 'AIFI_L2': '/home/jupyter/cache/cc78f20a-b962-4ddf-ae93-47d58741a644/ref_pbmc_clean_celltypist_model_AIFI_L2_2024-04-19.pkl',
 'AIFI_L3': '/home/jupyter/cache/671d1e43-bd32-4fea-bdda-d19a0484e664/ref_pbmc_clean_celltypist_model_AIFI_L3_2024-04-19.pkl'}

## Read sample metadata from HISE

In [13]:
sample_meta_file_uuid = 'cb01d5b1-40d8-4940-b61c-86d78de4528b'
file_query = hisepy.reader.read_files(
    [sample_meta_file_uuid]
)

In [14]:
meta_data = file_query['values']

In [15]:
meta_data.shape

(218, 33)

In [16]:
meta_data.head()

,Unnamed: 0,lastUpdated,sample.id,sample.bridgingControl,sample.sampleKitGuid,sample.visitName,sample.visitDetails,sample.drawDate,sample.daysSinceFirstVisit,file.id,...,subject.biologicalSex,subject.birthYear,subject.ethnicity,subject.partnerCode,subject.race,subject.subjectGuid,cohort.cohortGuid,specimens.specimenType,specimens.specimenGuid,filename
0,1,2024-08-14T19:23:45.883Z,3563c387-a91b-4584-b69d-1ebe3d7db641,False,KT00826,Flu Year 1 Day 90,N/A - Flu-Series Timepoint Only,2020-12-01T00:00:00Z,114,4aa6a6ac-e14c-4c41-9c44-9783c4252f20,...,Female,2009,Non-Hispanic origin,UP,African American,UP1001,UP1,PBMC,PB00826-07,218_UP1_Sample-HISEDescriptors/allUP-SDsamples...
1,2,2024-08-14T19:23:45.883Z,f9958635-c67f-4c2a-9c0c-32dc0fb21029,False,KT00891,Flu Year 1 Day 180-360,N/A - Flu-Series Timepoint Only,2021-03-01T00:00:00Z,217,c70d0dfe-4223-467f-998b-254247765793,...,Female,2009,Non-Hispanic origin,UP,African American,UP1001,UP1,Plasma,PL00891-20,218_UP1_Sample-HISEDescriptors/allUP-SDsamples...
2,3,2024-08-14T19:23:45.883Z,b8000280-a41f-492e-a5a7-a96df645daf7,False,KT00124,Flu Year 1 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2020-08-01T00:00:00Z,0,8eb7283b-0394-4ca9-8646-15ae4a206fe6,...,Female,2009,Non-Hispanic origin,UP,African American,UP1001,UP1,Tempus Tube,TP00124-01,218_UP1_Sample-HISEDescriptors/allUP-SDsamples...
3,4,2024-08-14T19:23:45.883Z,e2fad346-00ac-4492-abc6-f129cdcb6b76,False,KT00199,Flu Year 1 Day 7,N/A - Flu-Series Timepoint Only,2020-09-01T00:00:00Z,46,92945be5-51fc-4aea-9bcc-fde6bacd2301,...,Female,2009,Non-Hispanic origin,UP,African American,UP1001,UP1,NaN,PL00199-23,218_UP1_Sample-HISEDescriptors/allUP-SDsamples...
4,5,2024-08-14T19:23:45.883Z,56f49e95-9c67-4463-a4c7-7428c930c8e8,False,KT00943,COVID-19 Visit 3,N/A - Flu-Series Timepoint Only,2021-06-01T00:00:00Z,311,6db37504-2857-4774-8263-2331408cc405,...,Female,2009,Non-Hispanic origin,UP,African American,UP1001,UP1,PBMC,PB00943-11,218_UP1_Sample-HISEDescriptors/allUP-SDsamples...


## Apply across files

Here, we'll use `concurrent.futures` to apply the function above to our files in parallel.

In [17]:
out_dir = 'output'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

In [18]:
file_uuids = meta_data['file.id'].to_list()
sample_ids = meta_data['sample.sampleKitGuid'].to_list()

In [19]:
model_paths

{'AIFI_L1': '/home/jupyter/cache/482b9ec5-8631-48a1-a7ef-94e23fe97068/ref_pbmc_clean_celltypist_model_AIFI_L1_2024-04-18.pkl',
 'AIFI_L2': '/home/jupyter/cache/cc78f20a-b962-4ddf-ae93-47d58741a644/ref_pbmc_clean_celltypist_model_AIFI_L2_2024-04-19.pkl',
 'AIFI_L3': '/home/jupyter/cache/671d1e43-bd32-4fea-bdda-d19a0484e664/ref_pbmc_clean_celltypist_model_AIFI_L3_2024-04-19.pkl'}

In [20]:
print(len(file_uuids))
print(len(sample_ids))

218
218


In [57]:
# Process each subset in parallel
pool_executor = concurrent.futures.ProcessPoolExecutor(max_workers = 62)
with pool_executor as executor:
    
    futures = []
    for i in range(len(file_uuids)):
        file_uuid = file_uuids[i]
        sample_id = sample_ids[i]
        futures.append(executor.submit(process_data, file_uuid, sample_id, model_paths)) ##added model_paths arg

    # Check for errors when parallel processes return results
    for future in concurrent.futures.as_completed(futures):
        try:
            print(future.result())
        except Exception as e:
            print(f'Error: {e}')

🔬 Input data has 15674 cells and 33538 genes
🔗 Matching reference genes in the model
🧬 1109 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
🔬 Input data has 16634 cells and 33538 genes
🔗 Matching reference genes in the model
🧬 1109 features used for prediction
⚖️ Scaling input data
🔬 Input data has 15772 cells and 33538 genes
🔗 Matching reference genes in the model
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
🔬 Input data has 8633 cells and 33538 genes
🔗 Matching reference genes in the model
🧬 1109 features used for prediction
⚖️ Scaling input data
🧬 1109 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
🖋️ Predicting labels
✅ Prediction done!
👀 Can not dete

None


⛓️ Over-clustering input data with resolution set to 10
🗳️ Majority voting the predictions
✅ Majority voting done!
⛓️ Over-clustering input data with resolution set to 15
⛓️ Over-clustering input data with resolution set to 10
🔬 Input data has 11324 cells and 33538 genes
🔗 Matching reference genes in the model
⛓️ Over-clustering input data with resolution set to 10
⛓️ Over-clustering input data with resolution set to 10
⛓️ Over-clustering input data with resolution set to 15
⛓️ Over-clustering input data with resolution set to 15
⛓️ Over-clustering input data with resolution set to 10
⛓️ Over-clustering input data with resolution set to 15
🔬 Input data has 14394 cells and 33538 genes
🔗 Matching reference genes in the model
🔬 Input data has 15772 cells and 33538 genes
🔗 Matching reference genes in the model
⛓️ Over-clustering input data with resolution set to 15
⛓️ Over-clustering input data with resolution set to 10
🔬 Input data has 13496 cells and 33538 genes
🔗 Matching reference gene

None


🖋️ Predicting labels
🖋️ Predicting labels
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 10
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 10
🔬 Input data has 19076 cells and 33538 genes
🔗 Matching reference genes in the model
🗳️ Majority voting the predictions
✅ Majority voting done!
🗳️ Majority voting the predictions
✅ Majority voting done!
🗳️ Majority voting the predictions
✅ Majority voting done!
🧬 1936 features used for prediction
⚖️ Scaling input data
🗳️ Majority voting the predictions
✅ Majority voting done!
🗳️ Majority voting the predictions
🗳️ Majority voting the predictions
✅ Majority voting done!
✅ Majority voting done!
🗳️ Majority voting the predictions
✅ Majority voting done!
🖋️ Predicting labels
✅ Prediction done!
👀 Detected a neigh

...

⛓️ Over-clustering input data with resolution set to 10
🗳️ Majority voting the predictions
✅ Majority voting done!
🧬 1109 features used for prediction
⚖️ Scaling input data
🧬 2504 features used for prediction
⚖️ Scaling input data
🔬 Input data has 19692 cells and 33538 genes
🔗 Matching reference genes in the model
🧬 1936 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
⛓️ Over-clustering input data with resolution set to 10
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
🖋️ Predicting labels
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 10
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 10
🧬 1109 features used for prediction
⚖️ Scaling input data
🔬

None


🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
🗳️ Majority voting the predictions
✅ Majority voting done!
🧬 2504 features used for prediction
⚖️ Scaling input data
🔬 Input data has 22036 cells and 33538 genes
🔗 Matching reference genes in the model
🗳️ Majority voting the predictions
⛓️ Over-clustering input data with resolution set to 10
✅ Majority voting done!


None


🖋️ Predicting labels
⛓️ Over-clustering input data with resolution set to 10
✅ Prediction done!
👀 Detected a neighborhood graph in the input object, will run over-clustering on the basis of it
⛓️ Over-clustering input data with resolution set to 10
⛓️ Over-clustering input data with resolution set to 15
🧬 1109 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 10
🗳️ Majority voting the predictions
✅ Majority voting done!
🔬 Input data has 10687 cells and 33538 genes
🔗 Matching reference genes in the model
🔬 Input data has 21247 cells and 33538 genes
🔗 Matching reference genes in the model
🗳️ Majority voting the predictions
✅ Majority voting done!
⛓️ Over-clustering input data with resolution set to 10
🗳️ Majority voting the predictions
✅ Majority voting done!
🧬 1109 features used for prediction
⚖️ Scaling input 

## Assemble results

For each model, we'll assemble the results as a .csv file that we can utilize later for subclustering and analysis of major cell classes.

In [21]:
models = list(model_paths.keys())

In [22]:
models

['AIFI_L1', 'AIFI_L2', 'AIFI_L3']

In [23]:
out_files = []
for model in models:
    model_path = '/home/jupyter/ALL_UP1_analysis/Preprocessing/output/{m}'.format(m = model)
    model_path_files = os.listdir(model_path)
    model_files = []
    for model_path_file in model_path_files:
        if 'labels' in model_path_file:
            model_files.append(model_path_file)
    
    model_list = []

    for model_file in model_files:
        #print(model_file)
        df = pd.read_csv('/home/jupyter/ALL_UP1_analysis/Preprocessing/output/{m}/{f}'.format(m = model, f = model_file))
        model_list.append(df)
    model_df = pd.concat(model_list)

    out_csv = '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_{m}_{d}.csv'.format(
        m = model, d = date.today())
    out_files.append(out_csv)
    
    model_df.to_csv(out_csv)

    out_parquet = '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_{m}_{d}.parquet'.format(
        m = model, d = date.today())
    out_files.append(out_parquet)
    
    model_df.to_parquet(out_parquet)

In [62]:
model_df.shape

(3910928, 5)

In [24]:
model_df.shape

(3910928, 5)

In [25]:
model_df.head()

,barcodes,AIFI_L3,over_clustering,majority_voting,AIFI_L3_score
0,4cbf035202da11eda7895e7387a1b86f,Core naive CD4 T cell,211,Core naive CD4 T cell,1.000000
1,4cbf05aa02da11eda7895e7387a1b86f,GZMB- CD27+ EM CD4 T cell,85,CM CD4 T cell,1.000000
2,4cbf065402da11eda7895e7387a1b86f,Erythrocyte,0,Core naive CD4 T cell,0.999999
3,4cbf092402da11eda7895e7387a1b86f,Core CD14 monocyte,97,Core CD14 monocyte,1.000000
4,4cbf0c2602da11eda7895e7387a1b86f,Naive CD4 Treg,40,Naive CD4 Treg,1.000000


## Upload assembled data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [27]:
ss = hisepy.get_study_spaces()

In [28]:
ss

[{'id': '0b6bf907-6985-40e0-944d-677ac932677f',
  'accountGuid': 'b34b9c66-aa5d-45d0-9569-f78ca8f6813f',
  'projectGuid': '7ef501f4-79ea-45f0-ad97-6ed237cd5dd1',
  'driveId': '0AEgFtzSiOc7hUk9PVA',
  'auditInfo': {'added': '2024-08-14T20:42:25.668Z',
   'addedUser': 'temi.adewunmi@alleninstitute.org',
   'lastModified': '2024-08-19T17:19:23.424Z',
   'lastModifiedBy': 'temi.adewunmi@alleninstitute.org',
   'version': 1},
  'name': 'UP1 scRNAseq Study',
  'description': 'Project store for the work done on the scRNAseq samples from the UP1 cohort',
  'externalDrive': {'name': 'up1-studyExternal',
   'id': '1-wdxDzX3eTR2jYmdPEjJXE2HWtujRNWD',
   'parents': ['1NCAXL8ZmeQJawC2wRfG6si-O0CiMW3dN'],
   'webContentLink': '',
   'webViewLink': 'https://drive.google.com/drive/folders/1-wdxDzX3eTR2jYmdPEjJXE2HWtujRNWD',
   'iconLink': 'https://drive-thirdparty.googleusercontent.com/16/type/application/vnd.google-apps.folder+48+shared'},
  'reportDrive': {'name': 'up1-studyReport',
   'id': '1G9tr2

In [29]:
print(ss[0]['name'])
print(ss[0]['id'])
study_space_uuid = ss[0]['id']

title = '01 Raw scRNA-seq Assembly {d}'.format(d = date.today())

UP1 scRNAseq Study
0b6bf907-6985-40e0-944d-677ac932677f


In [30]:
title = '01 CellTypist Label Results {d}'.format(d = date.today())

In [31]:
search_id = element_id()
search_id

'cerium-nihonium-iodine'

In [32]:
len(meta_data['file.id'].to_list())

218

In [33]:
in_files = list(model_uuids.values()) + [sample_meta_file_uuid] + meta_data['file.id'].to_list() 

In [34]:
in_files[0:10]

['482b9ec5-8631-48a1-a7ef-94e23fe97068',
 'cc78f20a-b962-4ddf-ae93-47d58741a644',
 '671d1e43-bd32-4fea-bdda-d19a0484e664',
 'cb01d5b1-40d8-4940-b61c-86d78de4528b',
 '4aa6a6ac-e14c-4c41-9c44-9783c4252f20',
 'c70d0dfe-4223-467f-998b-254247765793',
 '8eb7283b-0394-4ca9-8646-15ae4a206fe6',
 '92945be5-51fc-4aea-9bcc-fde6bacd2301',
 '6db37504-2857-4774-8263-2331408cc405',
 '0a6567d8-6d7a-461e-b067-ee20f77d736f']

In [36]:
len(in_files)

222

In [35]:
out_files

['/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L1_2024-08-19.csv',
 '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L1_2024-08-19.parquet',
 '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L2_2024-08-19.csv',
 '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L2_2024-08-19.parquet',
 '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L3_2024-08-19.csv',
 '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L3_2024-08-19.parquet']

In [39]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

Cannot determine the current notebook.
1) /home/jupyter/ALL_UP1_analysis/Preprocessing/01-Python_label_predictions_celltypist.ipynb
2) /home/jupyter/ALL_UP1_analysis/Preprocessing/03-Python_assembly_of_subsets.ipynb
3) /home/jupyter/examples/Visualization_apps/dash/save_visualization_app_example.ipynb
Please select (1-3) 


 1


you are trying to upload file_ids... ['/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L1_2024-08-19.csv', '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L1_2024-08-19.parquet', '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L2_2024-08-19.csv', '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L2_2024-08-19.parquet', '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L3_2024-08-19.csv', '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L3_2024-08-19.parquet']. Do you truly want to proceed?


(y/n) y


{'trace_id': '5acbe432-ccba-4eba-a3fd-066b599c11f8',
 'files': ['/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L1_2024-08-19.csv',
  '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L1_2024-08-19.parquet',
  '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L2_2024-08-19.csv',
  '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L2_2024-08-19.parquet',
  '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L3_2024-08-19.csv',
  '/home/jupyter/ALL_UP1_analysis/Preprocessing/up1_celltypist_AIFI_L3_2024-08-19.parquet']}

In [56]:
import session_info
session_info.show()

#### Notes
1. hisepy cache_files vs read_files?
2. check number of labeled samples at the end of parallel processing
3. majority voting = True?
4. rename majority voting columns based on label level